# 钙钛矿稳定性预测案例研究
# Perovskite Stability Prediction Case Study

本Notebook演示如何预测卤化物钙钛矿太阳能电池的稳定性

**流程概览：**
1. 生成钙钛矿化学组成 (ABX₃)
2. Goldschmidt容忍因子计算
3. 八面体因子计算
4. 分解能DFT计算
5. 相变温度预测
6. 稳定性相图与可合成性评分

## 1. 环境设置与导入

In [ ]:
import os
import sys
import json
import numpy as np
import pandas as pd
from pathlib import Path
from datetime import datetime

# 导入自定义模块
sys.path.insert(0, '/root/.openclaw/workspace/dft_lammps_research')
sys.path.insert(0, '/root/.openclaw/workspace/dft_lammps_research/applications/perovskite')

from case_perovskite import PerovskiteStabilityConfig, PerovskiteStabilityAnalyzer

# 科学计算库
from pymatgen.core import Structure, Composition

# 可视化
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.colors import LinearSegmentedColormap

# 设置可视化样式
plt.style.use('seaborn-v0_8-whitegrid')

print("✓ 环境设置完成")
print(f"时间: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")

## 2. 配置参数设置

In [ ]:
# 创建配置
config = PerovskiteStabilityConfig(
    perovskite_type="halide",
    A_site_elements=["Cs", "Rb", "MA", "FA"],
    B_site_elements=["Pb", "Sn", "Ge"],
    X_site_elements=["I", "Br", "Cl"],
    tolerance_factor_range=(0.8, 1.0),
    max_decomposition_energy=0.1,
    work_dir="./perovskite_stability_results",
)

print("钙钛矿稳定性预测配置:")
print(f"  钙钛矿类型: {config.perovskite_type}")
print(f"  A位元素: {config.A_site_elements}")
print(f"  B位元素: {config.B_site_elements}")
print(f"  X位元素: {config.X_site_elements}")
print(f"  容忍因子范围: {config.tolerance_factor_range}")
print(f"  工作目录: {config.work_dir}")

## 3. Goldschmidt容忍因子简介

**Goldschmidt容忍因子**是预测钙钛矿结构稳定性的重要参数：

$$t = \\frac{r_A + r_X}{\\sqrt{2}(r_B + r_X)}$$

其中 $r_A$, $r_B$, $r_X$ 分别是A位、B位和X位离子的Shannon半径。

**结构预测规则：**
- $0.9 \leq t \leq 1.0$: 立方钙钛矿
- $0.8 \leq t \lt 0.9$: 正交/四方畸变
- $t \lt 0.8$ 或 $t \gt 1.1$: 非钙钛矿结构

## 4. Phase 1: 生成化学组成

In [ ]:
# 创建分析器
analyzer = PerovskiteStabilityAnalyzer(config)

# 生成化学组成
compositions = analyzer.generate_compositions()

print(f"\n生成 {len(compositions)} 个候选组成\n")

# 显示部分组成
df_comps = pd.DataFrame(compositions)
display(df_comps[['formula', 'A_site', 'B_site', 'X_site']].head(15))

## 5. Phase 2: 计算容忍因子和八面体因子

In [ ]:
# 计算容忍因子
stability_data = analyzer.calculate_tolerance_factors(compositions)

# 创建数据表
df_factors = pd.DataFrame(stability_data)

print("\n容忍因子计算结果:\n")
display(df_factors[['formula', 'tolerance_factor', 'octahedral_factor', 
                    'predicted_structure', 'synthesizability_score']].head(15))

## 6. 容忍因子-八面体因子相图

In [ ]:
# 创建容忍因子相图
fig, ax = plt.subplots(figsize=(14, 10))

x = df_factors['tolerance_factor']
y = df_factors['octahedral_factor']

# 按结构类型着色
structure_colors = {
    'cubic': '#2ecc71',
    'orthorhombic': '#3498db',
    'hexagonal': '#f39c12',
    'non-perovskite': '#e74c3c',
    'unknown': '#95a5a6'
}

for structure, color in structure_colors.items():
    mask = df_factors['predicted_structure'] == structure
    if mask.any():
        ax.scatter(x[mask], y[mask], s=200, label=structure, 
                  color=color, alpha=0.8, edgecolors='black', linewidth=1.5)

# 添加稳定区域边界
ax.axvline(x=0.8, color='gray', linestyle='--', linewidth=2, alpha=0.7, label='t=0.8')
ax.axvline(x=1.0, color='gray', linestyle='--', linewidth=2, alpha=0.7, label='t=1.0')
ax.axhline(y=0.442, color='gray', linestyle='--', linewidth=2, alpha=0.7, label='μ=0.442')

# 标记已知钙钛矿
known_perovskites = ['CsPbI3', 'CsPbBr3', 'MAPbI3', 'FAPbI3', 'CsSnI3']
for formula in known_perovskites:
    if formula in df_factors['formula'].values:
        row = df_factors[df_factors['formula'] == formula].iloc[0]
        ax.annotate(formula, (row['tolerance_factor'], row['octahedral_factor']),
                   xytext=(10, 10), textcoords='offset points',
                   fontsize=12, fontweight='bold',
                   bbox=dict(boxstyle='round,pad=0.4', facecolor='yellow', alpha=0.8))

# 添加区域标签
ax.text(0.9, 0.6, 'Stable\nPerovskite', fontsize=14, ha='center',
       fontweight='bold', color='darkgreen',
       bbox=dict(boxstyle='round', facecolor='lightgreen', alpha=0.5))
ax.text(0.7, 0.5, 'Non-perovskite', fontsize=12, ha='center',
       bbox=dict(boxstyle='round', facecolor='lightcoral', alpha=0.5))

ax.set_xlabel('Tolerance Factor (t)', fontsize=18, fontweight='bold')
ax.set_ylabel('Octahedral Factor (μ)', fontsize=18, fontweight='bold')
ax.set_title('Perovskite Stability Map\n(Goldschmidt Tolerance Factors)',
            fontsize=20, fontweight='bold', pad=20)
ax.set_xlim([0.6, 1.2])
ax.set_ylim([0.3, 0.9])
ax.legend(fontsize=12, loc='upper right')
ax.grid(True, alpha=0.3)
ax.tick_params(labelsize=14)

plt.tight_layout()
plt.savefig(f"{config.work_dir}/tolerance_factor_map_notebook.png", dpi=300, bbox_inches='tight')
plt.show()

print("✓ 容忍因子相图生成完成")

## 7. Phase 3: 计算分解能

In [ ]:
# 计算分解能
stability_results = analyzer.calculate_decomposition_energy(stability_data, skip_dft=True)

# 创建结果表
df_results = pd.DataFrame(stability_results)

print("\n分解能计算结果:\n")
display(df_results[['formula', 'decomposition_energy', 'is_stable', 
                    'stability_category']].head(15))

## 8. 分解能分布分析

In [ ]:
# 分解能分布图
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# 左图: 直方图
energies = df_results['decomposition_energy']

axes[0].hist(energies, bins=20, color='skyblue', edgecolor='black', alpha=0.7)
axes[0].axvline(x=0.02, color='green', linestyle='--', linewidth=2, label='Excellent')
axes[0].axvline(x=0.05, color='blue', linestyle='--', linewidth=2, label='Very Good')
axes[0].axvline(x=0.1, color='orange', linestyle='--', linewidth=2, label='Good')
axes[0].set_xlabel('Decomposition Energy (eV/atom)', fontsize=14, fontweight='bold')
axes[0].set_ylabel('Count', fontsize=14, fontweight='bold')
axes[0].set_title('Decomposition Energy Distribution', fontsize=16, fontweight='bold')
axes[0].legend(fontsize=12)
axes[0].grid(True, alpha=0.3, axis='y')

# 右图: 容忍因子 vs 分解能
x = df_results['tolerance_factor']
y = df_results['decomposition_energy']

colors = ['#2ecc71' if e <= 0.02 else '#3498db' if e <= 0.05 else '#f39c12' if e <= 0.1 else '#e74c3c' 
         for e in y]

axes[1].scatter(x, y, s=200, c=colors, alpha=0.8, edgecolors='black')
axes[1].axhline(y=0.1, color='red', linestyle='--', linewidth=2, alpha=0.7, label='Stability threshold')

# 标记已知钙钛矿
for formula in known_perovskites:
    if formula in df_results['formula'].values:
        row = df_results[df_results['formula'] == formula].iloc[0]
        axes[1].annotate(formula, (row['tolerance_factor'], row['decomposition_energy']),
                        xytext=(5, 5), textcoords='offset points', fontsize=9)

axes[1].set_xlabel('Tolerance Factor', fontsize=14, fontweight='bold')
axes[1].set_ylabel('Decomposition Energy (eV/atom)', fontsize=14, fontweight='bold')
axes[1].set_title('Stability vs Tolerance Factor', fontsize=16, fontweight='bold')
axes[1].legend(fontsize=12)
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(f"{config.work_dir}/decomposition_energy_analysis_notebook.png", dpi=300, bbox_inches='tight')
plt.show()

print("✓ 分解能分析完成")

## 9. Phase 4: 预测相变温度

In [ ]:
# 预测相变温度
final_results = analyzer.predict_phase_transition(stability_results)

# 创建最终结果表
df_final = pd.DataFrame(final_results)

print("\n相变温度预测结果:\n")
display(df_final[['formula', 'predicted_structure', 'phase_transition_temp', 
                  'room_temp_phase']].head(15))

## 10. 相变温度分布

In [ ]:
# 过滤有效温度数据
valid_temps = df_final[df_final['phase_transition_temp'].apply(lambda x: isinstance(x, (int, float)))]

if len(valid_temps) > 0:
    temps = valid_temps['phase_transition_temp'].astype(float)
    
    fig, ax = plt.subplots(figsize=(12, 6))
    
    ax.hist(temps, bins=15, color='coral', edgecolor='black', alpha=0.7)
    ax.axvline(x=298, color='blue', linestyle='--', linewidth=2, label='Room Temperature')
    ax.axvline(x=temps.mean(), color='red', linestyle='--', linewidth=2, 
              label=f'Mean: {temps.mean():.0f} K')
    
    ax.set_xlabel('Phase Transition Temperature (K)', fontsize=14, fontweight='bold')
    ax.set_ylabel('Count', fontsize=14, fontweight='bold')
    ax.set_title('Phase Transition Temperature Distribution', fontsize=16, fontweight='bold')
    ax.legend(fontsize=12)
    ax.grid(True, alpha=0.3, axis='y')
    
    plt.tight_layout()
    plt.savefig(f"{config.work_dir}/transition_temperature_distribution_notebook.png", dpi=300, bbox_inches='tight')
    plt.show()
    
    print(f"\n✓ 相变温度分布图生成完成")
    print(f"  平均相变温度: {temps.mean():.1f} K")
    print(f"  相变温度范围: {temps.min():.0f} - {temps.max():.0f} K")
else:
    print("无有效温度数据")

## 11. Phase 5: 综合分析与排名

In [ ]:
# 运行综合分析
df_analysis = analyzer.analyze_results(final_results)

print("\n钙钛矿稳定性排名:\n")
display(df_analysis[['Formula', 'Tolerance Factor', 'Decomposition Energy (eV/atom)', 
                     'Overall Stability Score', 'Recommendation']].head(15))

## 12. 元素组合稳定性矩阵

In [ ]:
# 创建元素组合稳定性矩阵
fig, axes = plt.subplots(1, 2, figsize=(18, 7))

# 左图: A-site vs B-site
pivot_ab = df_analysis.pivot_table(values='Overall Stability Score',
                                    index='A-site', columns='B-site', aggfunc='mean')
sns.heatmap(pivot_ab, annot=True, fmt='.2f', cmap='RdYlGn',
           ax=axes[0], vmin=0, vmax=1,
           cbar_kws={'label': 'Stability Score'})
axes[0].set_title('Stability Matrix: A-site vs B-site', fontsize=14, fontweight='bold')

# 右图: B-site vs X-site
pivot_bx = df_analysis.pivot_table(values='Overall Stability Score',
                                    index='B-site', columns='X-site', aggfunc='mean')
sns.heatmap(pivot_bx, annot=True, fmt='.2f', cmap='RdYlGn',
           ax=axes[1], vmin=0, vmax=1,
           cbar_kws={'label': 'Stability Score'})
axes[1].set_title('Stability Matrix: B-site vs X-site', fontsize=14, fontweight='bold')

plt.tight_layout()
plt.savefig(f"{config.work_dir}/element_stability_matrix_notebook.png", dpi=300, bbox_inches='tight')
plt.show()

print("✓ 元素组合稳定性矩阵生成完成")

## 13. Top 10稳定钙钛矿排名

In [ ]:
# Top 10排名图
fig, ax = plt.subplots(figsize=(12, 8))

top_10 = df_analysis.head(10)
colors = plt.cm.RdYlGn(top_10['Overall Stability Score'])

bars = ax.barh(range(len(top_10)), top_10['Overall Stability Score'],
              color=colors, edgecolor='black', linewidth=1.5)
ax.set_yticks(range(len(top_10)))
ax.set_yticklabels(top_10['Formula'], fontsize=13)
ax.set_xlabel('Overall Stability Score', fontsize=16, fontweight='bold')
ax.set_title('Top 10 Stable Perovskites', fontsize=18, fontweight='bold')
ax.invert_yaxis()
ax.set_xlim([0, 1])
ax.grid(True, alpha=0.3, axis='x')

# 添加数值标签
for i, (bar, score) in enumerate(zip(bars, top_10['Overall Stability Score'])):
    ax.text(score + 0.02, i, f'{score:.2f}', va='center', fontsize=11, fontweight='bold')

plt.tight_layout()
plt.savefig(f"{config.work_dir}/top10_ranking_notebook.png", dpi=300, bbox_inches='tight')
plt.show()

print("\n✓ Top 10排名图生成完成")
print(f"\n最佳候选: {top_10.iloc[0]['Formula']}")
print(f"  综合评分: {top_10.iloc[0]['Overall Stability Score']:.3f}")
print(f"  容忍因子: {top_10.iloc[0]['Tolerance Factor']:.3f}")
print(f"  分解能: {top_10.iloc[0]['Decomposition Energy (eV/atom)']:.3f} eV/atom")

## 14. 合成建议

In [ ]:
# 生成详细报告
analyzer.generate_report(df_analysis)

print("\n建议优先合成验证的钙钛矿:\n")
for i, (_, row) in enumerate(df_analysis.head(5).iterrows(), 1):
    print(f"{i}. {row['Formula']}")
    print(f"   - 容忍因子: {row['Tolerance Factor']:.3f}")
    print(f"   - 预测结构: {row['Predicted Structure']}")
    print(f"   - 分解能: {row['Decomposition Energy (eV/atom)']:.3f} eV/atom")
    print(f"   - 相变温度: {row['Phase Transition (K)']} K")
    print(f"   - 综合评分: {row['Overall Stability Score']:.3f}")
    print(f"   - 推荐等级: {row['Recommendation']}")
    print()

## 15. 总结

In [ ]:
print("="*70)
print("钙钛矿稳定性预测案例研究 - 完成摘要")
print("="*70)
print(f"\n总候选组成: {len(compositions)}")
print(f"稳定候选: {len(df_analysis[df_analysis['Decomposition Energy (eV/atom)'] <= 0.1])}")
print(f"立方相预测: {len(df_analysis[df_analysis['Predicted Structure'] == 'cubic'])}")

best = df_analysis.iloc[0]
print(f"\n最佳候选: {best['Formula']}")
print(f"  容忍因子: {best['Tolerance Factor']:.3f}")
print(f"  八面体因子: {best['Octahedral Factor']:.3f}")
print(f"  分解能: {best['Decomposition Energy (eV/atom)']:.3f} eV/atom")
print(f"  相变温度: {best['Phase Transition (K)']} K")
print(f"  室温相: {best['Room Temp Phase']}")
print(f"  综合评分: {best['Overall Stability Score']:.3f}")
print(f"  推荐等级: {best['Recommendation']}")

print(f"\n所有结果保存在: {Path(config.work_dir).absolute()}")
print("="*70)